# RAG Vector Benchmark - Simple Version

A beginner-friendly notebook to:
1. **Load data** from public datasets
2. **Create embeddings** (convert text to numbers)
3. **Store in Pinecone** (cloud vector database)
4. **Search with natural language**
5. **Run simple stress tests**

---

## What is RAG?

**RAG = Retrieval-Augmented Generation**

```
User asks: "How does Kubernetes work?"
        ↓
Step 1: Search your knowledge base for relevant docs
        ↓
Step 2: Give those docs + question to an LLM
        ↓
Result: Accurate answer based on YOUR data
```

## What is Pinecone?

- A **cloud database** for storing vectors (numbers that represent text)
- You search by meaning, not exact words
- No installation needed - just an API key
- Free tier: 100,000 vectors

---
## Step 0: Install Requirements

Run this once to install needed packages:

In [ ]:
# Uncomment and run this cell once to install packages
# !pip install pinecone-client sentence-transformers datasets tqdm python-dotenv matplotlib

---
## Step 1: Configuration

Get your **free Pinecone API key** at: https://www.pinecone.io/

1. Sign up (free)
2. Go to API Keys
3. Copy your key

In [ ]:
# ============================================
# CONFIGURATION - Edit these values!
# ============================================

PINECONE_API_KEY = "your-api-key-here"  # <-- PUT YOUR API KEY HERE!
INDEX_NAME = "rag-simple"               # Name for your Pinecone index
EMBEDDING_DIMENSION = 384               # Size of embeddings (don't change)

# How much data to load (keep small for testing)
NUM_DOCUMENTS = 100  # Number of text documents to load

print("Configuration set!")
print(f"  Index name: {INDEX_NAME}")
print(f"  Documents to load: {NUM_DOCUMENTS}")

---
## Step 2: Load Sample Data

We'll load real data from Wikipedia using HuggingFace datasets.

Each document has:
- `id`: Unique identifier
- `content`: The actual text
- `source`: Where it came from

In [ ]:
from datasets import load_dataset
from tqdm import tqdm

def load_wikipedia_data(num_docs=100):
    """
    Load Wikipedia articles from HuggingFace.
    Returns a list of documents with id, content, and source.
    """
    print(f"Loading {num_docs} Wikipedia articles...")
    
    # Load Simple English Wikipedia (smaller, easier to work with)
    dataset = load_dataset(
        "wikipedia",
        "20220301.simple",
        split="train",
        streaming=True,  # Don't download everything at once
        trust_remote_code=True
    )
    
    documents = []
    for item in tqdm(dataset, total=num_docs, desc="Loading"):
        if len(documents) >= num_docs:
            break
        
        # Get the text (first 1000 characters to keep it manageable)
        text = item.get("text", "")[:1000]
        title = item.get("title", "Unknown")
        
        if len(text) > 100:  # Skip very short articles
            documents.append({
                "id": f"wiki_{len(documents):04d}",
                "content": text,
                "title": title,
                "source": "wikipedia"
            })
    
    print(f"\nLoaded {len(documents)} documents!")
    return documents

# Load the data
documents = load_wikipedia_data(NUM_DOCUMENTS)

In [ ]:
# Let's look at a sample document
print("Sample Document:")
print("=" * 50)
print(f"ID: {documents[0]['id']}")
print(f"Title: {documents[0]['title']}")
print(f"Source: {documents[0]['source']}")
print(f"Content Preview:")
print(documents[0]['content'][:300] + "...")

---
## Step 3: Create Embeddings

**Embeddings** convert text into numbers (vectors) that capture meaning.

```
"I love dogs"  →  [0.23, -0.15, 0.89, ...]  (384 numbers)
"I adore puppies"  →  [0.21, -0.14, 0.87, ...]  (similar numbers!)
"Stock market crash"  →  [-0.45, 0.32, -0.11, ...]  (different numbers)
```

Similar meanings = similar numbers = we can find related content!

In [ ]:
from sentence_transformers import SentenceTransformer

# Load the embedding model (downloads ~100MB on first run)
print("Loading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded!")

def create_embedding(text):
    """
    Convert text to a vector (list of 384 numbers).
    """
    embedding = model.encode(text, normalize_embeddings=True)
    return embedding.tolist()

# Test it
test_embedding = create_embedding("Hello world")
print(f"\nEmbedding size: {len(test_embedding)} numbers")
print(f"First 5 values: {test_embedding[:5]}")

In [ ]:
# Create embeddings for all documents
print(f"Creating embeddings for {len(documents)} documents...")

# Get all content
texts = [doc['content'] for doc in documents]

# Create embeddings in batch (faster!)
embeddings = model.encode(texts, show_progress_bar=True, normalize_embeddings=True)

print(f"\nCreated {len(embeddings)} embeddings!")
print(f"Each embedding has {len(embeddings[0])} dimensions.")

---
## Step 4: Connect to Pinecone

Pinecone is a **cloud service** - your data is stored on their servers.

```
Your Computer  →  Internet  →  Pinecone Cloud (AWS/GCP)
     ↑                              ↑
  Your code                   Your vectors stored here
```

No Docker, no local database - just API calls!

In [ ]:
from pinecone import Pinecone, ServerlessSpec
import time

# Connect to Pinecone
print("Connecting to Pinecone...")
pc = Pinecone(api_key=PINECONE_API_KEY)

# Check if index exists
existing_indexes = [idx.name for idx in pc.list_indexes()]
print(f"Existing indexes: {existing_indexes}")

# Create index if it doesn't exist
if INDEX_NAME not in existing_indexes:
    print(f"\nCreating index '{INDEX_NAME}'...")
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBEDDING_DIMENSION,
        metric="cosine",  # How to measure similarity
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print("Waiting for index to be ready...")
    time.sleep(30)  # Wait for index creation

# Connect to the index
index = pc.Index(INDEX_NAME)
print(f"\nConnected to index: {INDEX_NAME}")

# Check stats
stats = index.describe_index_stats()
print(f"Current vectors in index: {stats.total_vector_count}")

---
## Step 5: Upload Data to Pinecone

We upload:
- **ID**: Unique identifier for each document
- **Values**: The embedding vector (384 numbers)
- **Metadata**: Original text and other info (so we can get it back!)

In [ ]:
def upload_to_pinecone(documents, embeddings, batch_size=50):
    """
    Upload documents and embeddings to Pinecone.
    """
    print(f"Uploading {len(documents)} vectors to Pinecone...")
    
    # Prepare vectors
    vectors = []
    for doc, emb in zip(documents, embeddings):
        vectors.append({
            "id": doc["id"],
            "values": emb.tolist(),
            "metadata": {
                "content": doc["content"][:500],  # Store first 500 chars
                "title": doc.get("title", ""),
                "source": doc["source"]
            }
        })
    
    # Upload in batches
    for i in tqdm(range(0, len(vectors), batch_size), desc="Uploading"):
        batch = vectors[i:i + batch_size]
        index.upsert(vectors=batch)
    
    # Wait for indexing
    print("Waiting for vectors to be indexed...")
    time.sleep(5)
    
    # Check stats
    stats = index.describe_index_stats()
    print(f"\nTotal vectors in index: {stats.total_vector_count}")

# Upload!
upload_to_pinecone(documents, embeddings)

---
## Step 6: Search with Natural Language!

Now the fun part! Ask questions and find relevant documents.

**How it works:**
1. Your question → embedding (384 numbers)
2. Pinecone finds vectors with similar numbers
3. Returns the matching documents with their content

In [ ]:
def search(query, top_k=5):
    """
    Search Pinecone with a natural language query.
    
    Args:
        query: Your question or search text
        top_k: Number of results to return
    """
    print(f"\n🔍 Searching for: '{query}'")
    print("-" * 50)
    
    # Step 1: Convert query to embedding
    query_embedding = create_embedding(query)
    
    # Step 2: Search Pinecone
    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True
    )
    
    # Step 3: Display results
    print(f"Found {len(results.matches)} results:\n")
    
    for i, match in enumerate(results.matches, 1):
        score = match.score
        title = match.metadata.get('title', 'Unknown')
        content = match.metadata.get('content', 'No content')
        
        print(f"[{i}] Score: {score:.3f} | {title}")
        print(f"    {content[:200]}...")
        print()
    
    return results

print("Search function ready!")
print("Usage: search('your question here')")

In [ ]:
# Try some searches!
search("space exploration and planets")

In [ ]:
search("famous scientists and their discoveries")

In [ ]:
search("animals and nature")

In [ ]:
# Try your own search!
MY_QUERY = "history and ancient civilizations"  # <-- Change this!
search(MY_QUERY)

---
## Step 7: Simple Stress Test

Let's test how fast Pinecone responds to multiple queries.

In [ ]:
import time
import statistics

def stress_test(num_queries=20):
    """
    Run multiple queries and measure response times.
    """
    print(f"Running {num_queries} test queries...\n")
    
    # Sample queries to test
    test_queries = [
        "science and technology",
        "history of the world",
        "animals and wildlife",
        "music and art",
        "sports and games",
        "food and cooking",
        "geography and countries",
        "mathematics and numbers",
        "medicine and health",
        "computers and internet"
    ]
    
    latencies = []
    
    for i in range(num_queries):
        # Pick a query
        query = test_queries[i % len(test_queries)]
        
        # Time the query
        start = time.time()
        query_embedding = create_embedding(query)
        results = index.query(vector=query_embedding, top_k=5)
        latency = (time.time() - start) * 1000  # Convert to milliseconds
        
        latencies.append(latency)
        print(f"  Query {i+1}: {latency:.1f}ms")
    
    # Calculate statistics
    print("\n" + "=" * 40)
    print("RESULTS")
    print("=" * 40)
    print(f"Total queries: {num_queries}")
    print(f"Average latency: {statistics.mean(latencies):.1f}ms")
    print(f"Min latency: {min(latencies):.1f}ms")
    print(f"Max latency: {max(latencies):.1f}ms")
    print(f"Std deviation: {statistics.stdev(latencies):.1f}ms")
    
    return latencies

# Run the test
latencies = stress_test(20)

---
## Step 8: Visualize Results

In [ ]:
import matplotlib.pyplot as plt

# Create a simple chart
plt.figure(figsize=(10, 4))

# Bar chart of latencies
plt.subplot(1, 2, 1)
plt.bar(range(len(latencies)), latencies, color='steelblue')
plt.xlabel('Query Number')
plt.ylabel('Latency (ms)')
plt.title('Query Latencies')
plt.axhline(y=statistics.mean(latencies), color='red', linestyle='--', label='Average')
plt.legend()

# Histogram
plt.subplot(1, 2, 2)
plt.hist(latencies, bins=10, color='steelblue', edgecolor='black')
plt.xlabel('Latency (ms)')
plt.ylabel('Count')
plt.title('Latency Distribution')

plt.tight_layout()
plt.savefig('stress_test_results.png', dpi=100)
plt.show()

print("\nChart saved as 'stress_test_results.png'")

---
## Step 9: Cleanup (Optional)

Delete the index when you're done to free up resources.

In [ ]:
# WARNING: This deletes all your data!
# Uncomment to run:

# pc.delete_index(INDEX_NAME)
# print(f"Deleted index: {INDEX_NAME}")

print("Index deletion is commented out for safety.")
print("Uncomment the line above if you want to delete.")

---
## Summary

You've learned:

1. **Load data** - Get text from public datasets
2. **Create embeddings** - Convert text to numbers with meaning
3. **Store in Pinecone** - Upload vectors to the cloud
4. **Search** - Find similar content with natural language
5. **Stress test** - Measure query performance

### Key Concepts

| Concept | What it is |
|---------|------------|
| **Embedding** | Numbers that represent text meaning |
| **Vector** | A list of numbers (like [0.1, -0.2, 0.3, ...]) |
| **Pinecone** | Cloud database for storing/searching vectors |
| **Cosine similarity** | How similar two vectors are (0 to 1) |
| **RAG** | Search + LLM for better answers |

### Next Steps

- Try loading different datasets (ArXiv, news, etc.)
- Add an LLM (OpenAI) to generate answers from retrieved docs
- Run concurrent stress tests
- Check out the full version in the `master` branch!

In [ ]:
print("\n" + "="*50)
print("🎉 You've completed the RAG Benchmark tutorial!")
print("="*50)
print(f"\nYour index '{INDEX_NAME}' has {index.describe_index_stats().total_vector_count} vectors.")
print("\nTry running: search('your own question')")